# 🎓 Machine Learning Project: Personalized AI Tutor via LLM & Reinforcement Learning
## 📌 Step 08: Reinforcement Learning Training (Q-Learning & DQN)

---

### 📖 Objective:
1. Implement Tabular Q-Learning with decaying $\epsilon$-greedy exploration and Bellman updates.
2. Train policy across 400 simulated student learning episodes.
3. Plot the **Reward Convergence Curve** and the **Learned Q-Table Value Heatmap**.


### Cell 1: Import Libraries & Define Q-Learning Agent


In [ ]:
# Cell 1: Imports & QLearningAgent Class
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

print("✅ Libraries Imported Successfully")

class QLearningAgent:
    def __init__(self, alpha=0.3, gamma=0.90, epsilon_start=1.0, epsilon_min=0.05, decay_rate=0.995):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.decay_rate = decay_rate
        self.q_table = np.zeros((10, 5))

    def get_state_idx(self, mastery_prob: float) -> int:
        return min(9, max(0, int(mastery_prob * 10)))

    def select_action(self, state_val: float, evaluation: bool = False) -> int:
        state_idx = self.get_state_idx(state_val)
        if not evaluation and np.random.rand() < self.epsilon:
            return np.random.randint(0, 5)
        return int(np.argmax(self.q_table[state_idx]))

    def update(self, state_val: float, action: int, reward: float, next_state_val: float):
        s, sp = self.get_state_idx(state_val), self.get_state_idx(next_state_val)
        td_target = reward + self.gamma * np.max(self.q_table[sp])
        self.q_table[s, action] += self.alpha * (td_target - self.q_table[s, action])
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.decay_rate


### Cell 2: Train Agent across 400 Student Episodes


In [ ]:
# Cell 2: RL Training Loop
class BKTTracker:
    def __init__(self): self.p_l = 0.25
    def update(self, correct: int) -> float:
        num = self.p_l * (0.9 if correct else 0.1)
        den = num + (1.0 - self.p_l) * (0.2 if correct else 0.8)
        self.p_l = float(np.clip((num / max(1e-6, den)) + 0.15, 0.01, 0.99))
        return self.p_l

class StudentSimulator:
    def __init__(self): self.skill = np.random.beta(2, 3)
    def step(self, diff):
        diff_norm = (diff - 1) / 4.0
        prob = 1.0 / (1.0 + np.exp(-4.0 * (self.skill - diff_norm)))
        corr = int(np.random.rand() < prob)
        gain = 0.06 * (1.2 if corr else 0.8) if abs(self.skill - diff_norm) < 0.25 else 0.02
        self.skill = min(1.0, self.skill + gain)
        reward = 1.6 * gain + 0.4 * corr + 0.4 - 0.25 * ((self.skill - diff_norm)**2)
        return corr, gain, reward

rl_agent = QLearningAgent()
rewards_log = []

for ep in range(400):
    student, bkt = StudentSimulator(), BKTTracker()
    ep_r = 0.0
    for _ in range(20):
        s = bkt.p_l
        a = rl_agent.select_action(s)
        corr, gain, r = student.step(a + 1)
        sp = bkt.update(corr)
        rl_agent.update(s, a, r, sp)
        ep_r += r
    rewards_log.append(ep_r)

print(f"✅ Training Finished Across 400 Episodes. Final Average Reward: {np.mean(rewards_log[-50:]):.3f}")


### Cell 3: Plot RL Convergence & Q-Table Heatmap


In [ ]:
# Cell 3: Plot Training Curves & Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

window = 20
smoothed = np.convolve(rewards_log, np.ones(window)/window, mode='valid')
axes[0].plot(rewards_log, color='#93c5fd', alpha=0.4, label='Raw Episode Reward')
axes[0].plot(range(window-1, len(rewards_log)), smoothed, color='#2563eb', lw=2.5, label='Moving Average')
axes[0].set_title('RL Policy Convergence Curve (400 Episodes)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Cumulative Reward')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

cax = axes[1].matshow(rl_agent.q_table, cmap='viridis', aspect='auto')
fig.colorbar(cax, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].set_title('Learned Q-Table Value Heatmap Q(s, a)', fontsize=12, fontweight='bold', pad=15)
axes[1].set_xlabel('Difficulty Action (Levels 1 to 5)')
axes[1].set_ylabel('Mastery State Bins (0 to 9)')
axes[1].set_xticks(range(5))
axes[1].set_xticklabels(['Lvl 1', 'Lvl 2', 'Lvl 3', 'Lvl 4', 'Lvl 5'])

plt.tight_layout()
plt.show()
print("✅ Step 08 Completed Successfully!")
